In [7]:
%pip install ucimlrepo pandas scikit-learn streamlit matplotlib seaborn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 140.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 131.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 132.5 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
  Attempting uninstall: numpy
    Found existing installation: nu

In [3]:
import os
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

os.makedirs("data", exist_ok=True)
os.makedirs("model", exist_ok=True)

load the data and rename columns

In [5]:
data = fetch_ucirepo(id=350)   # Default of Credit Card Clients
df = pd.concat([data.data.features, data.data.targets], axis=1)

df.columns = (["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]
              + [f"PAY_{i}" for i in range(1, 7)]
              + [f"BILL_AMT{i}" for i in range(1, 7)]
              + [f"PAY_AMT{i}" for i in range(1, 7)]
              + ["default_next_month"])

df.to_csv("data/credit_default_raw.csv", index=False)
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_next_month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


Inspect the data

In [13]:
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nTarget distribution:\n", df["default_next_month"].value_counts(normalize=True))
print("\nEDUCATION values:", sorted(df["EDUCATION"].unique()))
print("MARRIAGE values:", sorted(df["MARRIAGE"].unique()))
print("PAY_1 values:", sorted(df["PAY_1"].unique()))
df.dtypes

Shape: (30000, 24)
Missing values: 0
Duplicate rows: 35

Target distribution:
 default_next_month
0    0.7788
1    0.2212
Name: proportion, dtype: float64

EDUCATION values: [0, 1, 2, 3, 4, 5, 6]
MARRIAGE values: [0, 1, 2, 3]
PAY_1 values: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]


LIMIT_BAL             int64
SEX                   int64
EDUCATION             int64
MARRIAGE              int64
AGE                   int64
PAY_1                 int64
PAY_2                 int64
PAY_3                 int64
PAY_4                 int64
PAY_5                 int64
PAY_6                 int64
BILL_AMT1             int64
BILL_AMT2             int64
BILL_AMT3             int64
BILL_AMT4             int64
BILL_AMT5             int64
BILL_AMT6             int64
PAY_AMT1              int64
PAY_AMT2              int64
PAY_AMT3              int64
PAY_AMT4              int64
PAY_AMT5              int64
PAY_AMT6              int64
default_next_month    int64
dtype: object

Cleaning

In [8]:
# Undocumented category codes are merged into the "other" category
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})   # 4 = others
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})                 # 3 = others

# Remove duplicate rows so copies can't land in both train and test
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows, final shape: {df.shape}")

print(sorted(df["EDUCATION"].unique()), sorted(df["MARRIAGE"].unique()))

Dropped 35 duplicate rows, final shape: (29965, 24)
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)] [np.int64(1), np.int64(2), np.int64(3)]


Spliting

In [10]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.25
SEED = 8531         

TARGET = "default_next_month"
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(X_train.shape, X_test.shape)
print("Train default rate:", round(y_train.mean(), 4), "| Test default rate:", round(y_test.mean(), 4))

# This is the file you upload to the Streamlit app and commit to GitHub
test_df = X_test.copy()
test_df[TARGET] = y_test
test_df.to_csv("test_data.csv", index=False)

(22473, 23) (7492, 23)
Train default rate: 0.2212 | Test default rate: 0.2213


Define the Models

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

models = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=SEED),
    "KNN": Pipeline([
        ("scale", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=15)),
    ]),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": Pipeline([
        ("scale", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, class_weight="balanced")),
    ]),
}

Train, Evaluate, Save

In [14]:
import joblib, time
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def evaluate(model, X, y):
    pred = model.predict(X)
    return {
        "Accuracy": accuracy_score(y, pred),
        "AUC": roc_auc_score(y, get_scores(model, X)),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred),
        "F1": f1_score(y, pred),
        "MCC": matthews_corrcoef(y, pred),
    }

results = {}
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    results[name] = evaluate(model, X_test, y_test)
    fname = "model/" + name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(model, fname)
    print(f"{name}: trained in {time.time() - start:.1f}s, saved to {fname}")

results_df = pd.DataFrame(results).T.round(4)
results_df.to_csv("model_comparison.csv")
results_df

Logistic Regression: trained in 1.5s, saved to model/logistic_regression.joblib
Decision Tree: trained in 0.3s, saved to model/decision_tree.joblib
KNN: trained in 1.7s, saved to model/knn.joblib
Naive Bayes: trained in 0.1s, saved to model/naive_bayes.joblib
Support Vector Machine: trained in 63.2s, saved to model/support_vector_machine.joblib


,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.6945,0.7352,0.3889,0.6665,0.4912,0.3156
Decision Tree,0.7419,0.7668,0.4445,0.6665,0.5333,0.3789
KNN,0.8125,0.7468,0.6511,0.3287,0.4369,0.3672
Naive Bayes,0.3684,0.6704,0.2447,0.8884,0.3837,0.1139
Support Vector Machine,0.7835,0.7711,0.5092,0.6001,0.5509,0.4120
